In [1]:
import struct
def bf16_to_float_block(bf16_block):
    # 左移16位填充为32位表示
    float_block = []
    for bf16 in bf16_block:
        fp32_bits = bf16 << 16
        float_block.append(struct.unpack('>f', struct.pack('>I', fp32_bits))[0])
    # 转换为浮点数
    return float_block

bf16_to_float_block([16960, 16704, 16736, 16704])

[48.0, 12.0, 14.0, 12.0]

In [ ]:
padding = {-2: []}
if -2 in padding.keys():
    print(True)
padding.keys()

True


dict_keys([-2])

In [2]:
a = [2, 3]
a.reverse()
a

[3, 2]

In [3]:
bf16_to_float_block([17094, 17032])

[99.0, 68.0]

In [2]:
import numpy as np
np.array([])

array([], dtype=float64)

In [6]:
[[] for _ in range(4)]

[[], [], [], []]

In [4]:
num = 1
bin(num)[2:].zfill(2)

'01'

In [4]:
import numpy as np

a = np.array(1)
b = a.copy()

b = np.array(2)
print(a)
print(b)

1
2


In [5]:
a = [[], []]
a[1]

[]

In [1]:
hbm = True
not hbm

False

In [24]:
queue = [1, 2, 3]
queue.pop(0)

1

In [21]:
a = [[]] * 2
print(a)
a[0].append(1)
print(a)
b = [[0], []]
b


[[], []]
[[1], [1]]


[[0], []]

In [1]:
import numpy as np
from scipy.sparse import csr_matrix

D = np.array([[1, 1, 1, 0], [0, 0, 0, 0], [0, 1, 1, 0]])
csr = csr_matrix(D)

print("Values:", csr.data)
print("Column indices:", csr.indices)
print("Row pointers:", csr.indptr)

Values: [1 1 1 1 1]
Column indices: [0 1 2 1 2]
Row pointers: [0 3 3 5]


In [2]:
import numpy as np
from scipy.sparse import csr_matrix

D = np.array([[1, 1, 1, 1]])
csr = csr_matrix(D)

print("Values:", csr.data)
print("Column indices:", csr.indices)
print("Row pointers:", csr.indptr)

Values: [1 1 1 1]
Column indices: [0 1 2 3]
Row pointers: [0 4]


In [3]:
def store_csr_in_simple_blocks(csr_matrix, elements_per_block=2):
    """
    将CSR矩阵简单分块存储，每块包含固定数量的values和indices

    Args:
        csr_matrix: 输入的CSR格式矩阵
        elements_per_block: 每块包含的元素数量
    """
    values = csr_matrix.data
    col_indices = csr_matrix.indices
    row_pointers = csr_matrix.indptr
    num_rows = csr_matrix.shape[0]
    total_values = len(values)

    # 计算需要的块数量
    num_blocks = (total_values + elements_per_block - 1) // elements_per_block

    blocks = []

    # 对每个块进行处理
    for block_idx in range(num_blocks):
        start_idx = block_idx * elements_per_block
        end_idx = min(start_idx + elements_per_block, total_values)

        # 提取当前块的values和indices
        block_values = values[start_idx:end_idx].tolist()
        block_col_indices = col_indices[start_idx:end_idx].tolist()

        # 找出当前块涉及的行范围
        # 对每行检查是否有元素在当前块的范围内
        rows_in_block = []
        for row in range(num_rows):
            row_start_pos = row_pointers[row]
            row_end_pos = row_pointers[row + 1]

            # 如果行的元素范围与块范围有重叠
            if row_start_pos < end_idx and row_end_pos > start_idx:
                rows_in_block.append(row)

        if not rows_in_block:
            print(f"警告: 块 {block_idx} 没有找到相关行!")
            continue

        row_start = min(rows_in_block)

        # 创建块内行指针
        block_row_ptr = [0]
        current_pos = 0

        # 计算每行在块内的元素数量
        for row in range(row_start, max(rows_in_block) + 1):
            if row in rows_in_block:
                # 计算该行在当前块内的元素数量
                row_begin = max(row_pointers[row], start_idx)
                row_end_pos = min(row_pointers[row + 1], end_idx)
                row_elements = max(0, row_end_pos - row_begin)
            else:
                # 块内不存在的行
                row_elements = 0

            current_pos += row_elements
            block_row_ptr.append(current_pos)

        # 创建块
        block = {
            "values": block_values,
            "col_indices": block_col_indices,
            "row_ptr": block_row_ptr,
            "row_start_index": row_start,
        }

        blocks.append(block)

    return blocks


def test_simple_blocks():
    """测试简化的分块策略"""
    D = np.array([[1, 1, 1, 0], [0, 0, 0, 0], [0, 1, 1, 0]])
    csr = csr_matrix(D)

    print("原始CSR数据:")
    print("Values:", csr.data)
    print("Column indices:", csr.indices)
    print("Row pointers:", csr.indptr)

    # 按每块2个元素分块
    blocks = store_csr_in_simple_blocks(csr, elements_per_block=2)

    # 显示分块结果
    print("\n简化分块结果:")
    for i, block in enumerate(blocks):
        print(f"\n块 #{i}:")
        print(f"  row_start_index: {block['row_start_index']}")
        print(f"  values: {block['values']}")
        print(f"  column: {block['col_indices']}")
        print(f"  row_ptr: {block['row_ptr']}")

    # 验证是否符合预期
    expected_blocks = [
        {
            "row_start_index": 0,
            "values": [1, 1],
            "col_indices": [0, 1],
            "row_ptr": [0, 2],
        },
        {
            "row_start_index": 0,
            "values": [1, 1],
            "col_indices": [2, 1],
            "row_ptr": [0, 1, 1, 2],
        },
        {"row_start_index": 2, "values": [1], "col_indices": [2], "row_ptr": [0, 1]},
    ]

    all_match = True
    for i, (actual, expected) in enumerate(zip(blocks, expected_blocks)):
        block_match = (
            actual["row_start_index"] == expected["row_start_index"]
            and actual["values"] == expected["values"]
            and actual["col_indices"] == expected["col_indices"]
            and actual["row_ptr"] == expected["row_ptr"]
        )

        print(f"\n块 #{i} {'匹配预期' if block_match else '不匹配预期'}!")
        if not block_match:
            print("预期:")
            print(f"  row_start_index: {expected['row_start_index']}")
            print(f"  values: {expected['values']}")
            print(f"  column: {expected['col_indices']}")
            print(f"  row_ptr: {expected['row_ptr']}")

            print("实际:")
            print(f"  row_start_index: {actual['row_start_index']}")
            print(f"  values: {actual['values']}")
            print(f"  column: {actual['col_indices']}")
            print(f"  row_ptr: {actual['row_ptr']}")

            all_match = False

    print("\n总体结果: " + ("全部匹配预期! ✓" if all_match else "存在不匹配 ✗"))

    return blocks

test_simple_blocks()

原始CSR数据:
Values: [1 1 1 1 1]
Column indices: [0 1 2 1 2]
Row pointers: [0 3 3 5]

简化分块结果:

块 #0:
  row_start_index: 0
  values: [1, 1]
  column: [0, 1]
  row_ptr: [0, np.int32(2)]

块 #1:
  row_start_index: 0
  values: [1, 1]
  column: [2, 1]
  row_ptr: [0, np.int32(1), np.int32(1), np.int32(2)]

块 #2:
  row_start_index: 2
  values: [1]
  column: [2]
  row_ptr: [0, np.int32(1)]

块 #0 匹配预期!

块 #1 匹配预期!

块 #2 匹配预期!

总体结果: 全部匹配预期! ✓


[{'values': [1, 1],
  'col_indices': [0, 1],
  'row_ptr': [0, np.int32(2)],
  'row_start_index': 0},
 {'values': [1, 1],
  'col_indices': [2, 1],
  'row_ptr': [0, np.int32(1), np.int32(1), np.int32(2)],
  'row_start_index': 0},
 {'values': [1],
  'col_indices': [2],
  'row_ptr': [0, np.int32(1)],
  'row_start_index': 2}]

In [ ]:
def store_csr_in_blocks(csr_matrix, block_size=2048):
    """将CSR矩阵分块存储在2KB块中"""
    # 提取CSR数据
    values = csr_matrix.data  # 16位值
    col_indices = csr_matrix.indices  # 16位列索引
    row_pointers = csr_matrix.indptr  # 32位行指针

    # 计算每块的元素数量
    values_per_block = 256
    col_indices_per_block = 256
    row_pointers_per_block = 256

    # 计算需要的块数量
    total_values = len(values)
    total_col_indices = len(col_indices)
    total_row_pointers = len(row_pointers)

    blocks_for_values = (total_values + values_per_block - 1) // values_per_block
    blocks_for_indices = (
        total_col_indices + col_indices_per_block - 1
    ) // col_indices_per_block
    blocks_for_pointers = (
        total_row_pointers + row_pointers_per_block - 1
    ) // row_pointers_per_block

    total_blocks = max(blocks_for_values, blocks_for_indices, blocks_for_pointers)

    # 创建块数组
    blocks = []

    # 填充每个块
    for block_idx in range(total_blocks):
        # 确定要复制的数据范围
        val_start = block_idx * values_per_block
        val_end = min(val_start + values_per_block, total_values)

        col_start = block_idx * col_indices_per_block
        col_end = min(col_start + col_indices_per_block, total_col_indices)

        row_start = block_idx * row_pointers_per_block
        row_end = min(row_start + row_pointers_per_block, total_row_pointers)

        # 创建块
        block = {
            "values": (
                values[val_start:val_end].tolist() if val_start < total_values else []
            ),
            "col_indices": (
                col_indices[col_start:col_end].tolist()
                if col_start < total_col_indices
                else []
            ),
            "row_pointers": (
                row_pointers[row_start:row_end].tolist()
                if row_start < total_row_pointers
                else []
            ),
        }

        # 填充到固定大小
        if len(block["values"]) < values_per_block:
            block["values"].extend([0] * (values_per_block - len(block["values"])))

        if len(block["col_indices"]) < col_indices_per_block:
            block["col_indices"].extend(
                [0] * (col_indices_per_block - len(block["col_indices"]))
            )

        if len(block["row_pointers"]) < row_pointers_per_block:
            block["row_pointers"].extend(
                [0] * (row_pointers_per_block - len(block["row_pointers"]))
            )

        blocks.append(block)

    return blocks

In [7]:
import numpy as np
from scipy.sparse import csr_matrix
import time


def store_csr_in_blocks(
    csr_matrix,
    values_per_block=256,
    col_indices_per_block=256,
    row_pointers_per_block=256,
):
    """将CSR矩阵分块存储在2KB块中"""
    # 提取CSR数据
    values = csr_matrix.data  # 16位值
    col_indices = csr_matrix.indices  # 16位列索引
    row_pointers = csr_matrix.indptr  # 32位行指针

    # 计算需要的块数量
    total_values = len(values)
    total_col_indices = len(col_indices)
    total_row_pointers = len(row_pointers)

    blocks_for_values = (total_values + values_per_block - 1) // values_per_block
    blocks_for_indices = (
        total_col_indices + col_indices_per_block - 1
    ) // col_indices_per_block
    blocks_for_pointers = (
        total_row_pointers + row_pointers_per_block - 1
    ) // row_pointers_per_block

    total_blocks = max(blocks_for_values, blocks_for_indices, blocks_for_pointers)

    print(f"总块数: {total_blocks}")
    print(f"Values需要块数: {blocks_for_values}")
    print(f"Indices需要块数: {blocks_for_indices}")
    print(f"Row Pointers需要块数: {blocks_for_pointers}")

    # 创建块数组
    blocks = []

    # 填充每个块
    for block_idx in range(total_blocks):
        # 确定要复制的数据范围
        val_start = block_idx * values_per_block
        val_end = min(val_start + values_per_block, total_values)

        col_start = block_idx * col_indices_per_block
        col_end = min(col_start + col_indices_per_block, total_col_indices)

        row_start = block_idx * row_pointers_per_block
        row_end = min(row_start + row_pointers_per_block, total_row_pointers)

        # 创建块
        block = {
            "values": (
                values[val_start:val_end].tolist() if val_start < total_values else []
            ),
            "col_indices": (
                col_indices[col_start:col_end].tolist()
                if col_start < total_col_indices
                else []
            ),
            "row_pointers": (
                row_pointers[row_start:row_end].tolist()
                if row_start < total_row_pointers
                else []
            ),
        }

        # 填充到固定大小 (在真实场景中可能使用特殊标记)
        if len(block["values"]) < values_per_block:
            block["values"].extend([None] * (values_per_block - len(block["values"])))

        if len(block["col_indices"]) < col_indices_per_block:
            block["col_indices"].extend(
                [None] * (col_indices_per_block - len(block["col_indices"]))
            )

        if len(block["row_pointers"]) < row_pointers_per_block:
            block["row_pointers"].extend(
                [None] * (row_pointers_per_block - len(block["row_pointers"]))
            )

        blocks.append(block)

    return blocks


def reconstruct_csr_from_blocks(
    blocks, num_rows, num_cols, values_per_block=256, col_indices_per_block=256
):
    """从块重建CSR矩阵"""
    # 合并数据
    all_values = []
    all_col_indices = []
    all_row_pointers = []

    for block in blocks:
        # 提取非填充数据
        real_values = [v for v in block["values"] if v is not None]
        real_indices = [i for i in block["col_indices"] if i is not None]
        real_pointers = [p for p in block["row_pointers"] if p is not None]

        all_values.extend(real_values)
        all_col_indices.extend(real_indices)
        all_row_pointers.extend(real_pointers)

    # 确保行指针正确完整
    if len(all_row_pointers) < num_rows + 1:
        print("错误: 重建的行指针数量不足!")

    # 重建CSR矩阵
    reconstructed_csr = csr_matrix(
        (all_values, all_col_indices, all_row_pointers), shape=(num_rows, num_cols)
    )

    return reconstructed_csr


def calculate_block_size_bytes(
    values_per_block=256, col_indices_per_block=256, row_pointers_per_block=256
):
    """计算块的总大小(字节)"""
    values_size = values_per_block * 2  # 16位 = 2字节
    col_indices_size = col_indices_per_block * 2  # 16位 = 2字节
    row_pointers_size = row_pointers_per_block * 4  # 32位 = 4字节

    total_size = values_size + col_indices_size + row_pointers_size
    return total_size


def verify_csr_blocks(original_csr, reconstructed_csr):
    """验证重建的CSR矩阵是否与原始矩阵相同"""
    # 检查形状
    if original_csr.shape != reconstructed_csr.shape:
        print(
            f"形状不匹配! 原始: {original_csr.shape}, 重建: {reconstructed_csr.shape}"
        )
        return False

    # 检查非零元素数量
    if len(original_csr.data) != len(reconstructed_csr.data):
        print(
            f"非零元素数量不匹配! 原始: {len(original_csr.data)}, 重建: {len(reconstructed_csr.data)}"
        )
        return False

    # 检查数据值
    if not np.array_equal(original_csr.data, reconstructed_csr.data):
        print("Values不匹配!")
        print(f"原始: {original_csr.data}")
        print(f"重建: {reconstructed_csr.data}")
        return False

    # 检查列索引
    if not np.array_equal(original_csr.indices, reconstructed_csr.indices):
        print("Column indices不匹配!")
        print(f"原始: {original_csr.indices}")
        print(f"重建: {reconstructed_csr.indices}")
        return False

    # 检查行指针
    if not np.array_equal(original_csr.indptr, reconstructed_csr.indptr):
        print("Row pointers不匹配!")
        print(f"原始: {original_csr.indptr}")
        print(f"重建: {reconstructed_csr.indptr}")
        return False

    # 检查转换为密集矩阵后是否相同
    if not np.array_equal(original_csr.toarray(), reconstructed_csr.toarray()):
        print("转换为密集矩阵后不匹配!")
        return False

    return True


def test_csr_blocking(
    matrix, values_per_block=256, col_indices_per_block=256, row_pointers_per_block=256
):
    """测试CSR矩阵分块存储和重建"""
    print(f"\n测试矩阵形状: {matrix.shape}")

    # 转换为CSR格式
    csr = csr_matrix(matrix)

    print(f"非零元素: {len(csr.data)}")
    print(
        f"Values: {csr.data[:10]}..." if len(csr.data) > 10 else f"Values: {csr.data}"
    )
    print(
        f"Column indices: {csr.indices[:10]}..."
        if len(csr.indices) > 10
        else f"Column indices: {csr.indices}"
    )
    print(
        f"Row pointers: {csr.indptr[:10]}..."
        if len(csr.indptr) > 10
        else f"Row pointers: {csr.indptr}"
    )

    block_size = calculate_block_size_bytes(
        values_per_block, col_indices_per_block, row_pointers_per_block
    )
    print(f"块大小: {block_size} 字节 ({block_size/1024:.2f} KB)")

    # 分块存储
    start_time = time.time()
    blocks = store_csr_in_blocks(
        csr, values_per_block, col_indices_per_block, row_pointers_per_block
    )
    blocking_time = time.time() - start_time
    print(f"分块时间: {blocking_time:.6f} 秒")

    # 打印第一个块的内容
    if blocks:
        print("\n第一个块的内容:")
        print(
            f"Values: {blocks[0]['values'][:5]}..."
            if blocks[0]["values"]
            else "Values: []"
        )
        print(
            f"Column indices: {blocks[0]['col_indices'][:5]}..."
            if blocks[0]["col_indices"]
            else "Column indices: []"
        )
        print(
            f"Row pointers: {blocks[0]['row_pointers'][:5]}..."
            if blocks[0]["row_pointers"]
            else "Row pointers: []"
        )

    # 重建CSR
    start_time = time.time()
    reconstructed_csr = reconstruct_csr_from_blocks(
        blocks, csr.shape[0], csr.shape[1], values_per_block, col_indices_per_block
    )
    reconstruction_time = time.time() - start_time
    print(f"重建时间: {reconstruction_time:.6f} 秒")

    # 验证
    is_valid = verify_csr_blocks(csr, reconstructed_csr)
    print(f"\n验证结果: {'成功' if is_valid else '失败'}")

    return is_valid, blocks


# 测试用例
def run_tests():
    print("===== 测试CSR矩阵分块存储 =====")

    # 测试1: 小矩阵
    print("\n测试1: 小矩阵")
    small_matrix = np.array([[1, 1, 1, 0], [0, 1, 1, 0]])
    test_csr_blocking(small_matrix)

    # 测试2: 中等大小矩阵
    print("\n测试2: 中等大小矩阵")
    medium_matrix = np.random.choice([0, 1], size=(100, 100), p=[0.95, 0.05])
    test_csr_blocking(medium_matrix)

    # 测试3: 大矩阵(在计算资源允许的情况下)
    print("\n测试3: 较大矩阵")
    large_matrix = np.random.choice([0, 1], size=(4096, 4096), p=[0.9, 0.1])
    test_csr_blocking(large_matrix)

    # 测试4: 自定义块大小
    print("\n测试4: 自定义块大小 (128-128-64)")
    test_csr_blocking(medium_matrix, 128, 128, 64)


# 运行测试
run_tests()

===== 测试CSR矩阵分块存储 =====

测试1: 小矩阵

测试矩阵形状: (2, 4)
非零元素: 5
Values: [1 1 1 1 1]
Column indices: [0 1 2 1 2]
Row pointers: [0 3 5]
块大小: 2048 字节 (2.00 KB)
总块数: 1
Values需要块数: 1
Indices需要块数: 1
Row Pointers需要块数: 1
分块时间: 0.000019 秒

第一个块的内容:
Values: [1, 1, 1, 1, 1]...
Column indices: [0, 1, 2, 1, 2]...
Row pointers: [0, 3, 5, None, None]...
重建时间: 0.000050 秒

验证结果: 成功

测试2: 中等大小矩阵

测试矩阵形状: (100, 100)
非零元素: 494
Values: [1 1 1 1 1 1 1 1 1 1]...
Column indices: [20 56 94  5 17 25 27 28 31 37]...
Row pointers: [ 0  3 17 21 25 28 33 38 44 46]...
块大小: 2048 字节 (2.00 KB)
总块数: 2
Values需要块数: 2
Indices需要块数: 2
Row Pointers需要块数: 1
分块时间: 0.000040 秒

第一个块的内容:
Values: [1, 1, 1, 1, 1]...
Column indices: [20, 56, 94, 5, 17]...
Row pointers: [0, 3, 17, 21, 25]...
重建时间: 0.000281 秒

验证结果: 成功

测试3: 较大矩阵

测试矩阵形状: (4096, 4096)
非零元素: 1677310
Values: [1 1 1 1 1 1 1 1 1 1]...
Column indices: [  0   5   9  14  72  92  95  96 123 132]...
Row pointers: [   0  391  832 1226 1646 2020 2419 2835 3227 3666]...
块大小: 2048 字节 (2.0

In [2]:
from distribution_sim import bf16_to_float

print(bf16_to_float(16256))
print(bf16_to_float(16384))

1.0
2.0
